# Structured Prompts

The `structured.py` module defines a structured chat prompt that combines chat messages with an output schema. It is designed to be composed with a language model that supports structured output.

# StructuredPrompt: `ChatPromptTemplate`

`StructuredPrompt` is a beta chat prompt template that stores an output schema and automatically configures a compatible language model to generate structured output.

## Fields

1. `schema_`:`dict[str, Any] | type`:= Stores the schema that defines the expected structured output.

2. `structured_output_kwargs`:`dict[str, Any]`:= Stores additional arguments passed to the language model's `with_structured_output()` method. Its default value is an empty dictionary.

   **Syntax**

   ```python
   structured_output_kwargs: dict[str, Any] = Field(
       default_factory=dict # Create a separate empty dictionary for each instance
   )
   ```

## Methods

1. `__init__`:= Creates a structured prompt from message representations and a required output schema.

   A `ValueError` is raised when the schema is missing or empty. Extra keyword arguments that are not recognised Pydantic fields are moved into `structured_output_kwargs`.

   **Syntax**

   ```python
   __init__(
       self, # Structured prompt instance
       messages: Sequence[MessageLikeRepresentation], # Message representations used by the prompt
       schema_: dict[str, Any] | type[BaseModel] | None = None, # Structured output schema
       *,
       structured_output_kwargs: dict[str, Any] | None = None, # Arguments for structured output generation
       template_format: PromptTemplateFormat = "f-string", # Format used by string message templates
       **kwargs: Any # Additional prompt or structured-output arguments
   ) -> None
   ```

2. `get_lc_namespace`:= Returns the LangChain serialization namespace assigned to the class.

   **Syntax**

   ```python
   @classmethod
   get_lc_namespace(
       cls # StructuredPrompt class
   ) -> list[str]
   ```

3. `from_messages_and_schema`:= Creates a structured prompt from message representations and an output schema.

   Additional keyword arguments are passed to the language model's structured-output configuration.

   **Syntax**

   ```python
   @classmethod
   from_messages_and_schema(
       cls, # StructuredPrompt class
       messages: Sequence[MessageLikeRepresentation], # Message representations used by the prompt
       schema: dict[str, Any] | type, # Dictionary schema or Pydantic model
       **kwargs: Any # Additional structured-output arguments
   ) -> ChatPromptTemplate
   ```

4. `__or__`:= Pipes the structured prompt into another Runnable-compatible object.

   This method delegates composition to `pipe()`.

   **Syntax**

   ```python
   __or__(
       self, # Structured prompt instance
       other: Runnable[PromptValue, Other]
       | Callable[[Iterator[PromptValue]], Iterator[Other]]
       | Callable[[AsyncIterator[PromptValue]], AsyncIterator[Other]]
       | Callable[[PromptValue], Other]
       | Mapping[
           str,
           Runnable[PromptValue, Any]
           | Callable[[PromptValue], Any]
           | Any
       ] # Runnable, function, or mapping to compose with the prompt
   ) -> RunnableSerializable[dict[str, Any], Any]
   ```

5. `pipe`:= Pipes the structured prompt into a language model and creates a `RunnableSequence`.

   The first supplied object must be a `BaseLanguageModel` or expose a `with_structured_output()` method. The stored schema and `structured_output_kwargs` are applied automatically.

   A `NotImplementedError` is raised when the first object does not support structured output.

   **Syntax**

   ```python
   pipe(
       self, # Structured prompt instance
       *others: Runnable[Any, Other]
       | Callable[[Iterator[Any]], Iterator[Other]]
       | Callable[[AsyncIterator[Any]], AsyncIterator[Other]]
       | Callable[[Any], Other]
       | Mapping[
           str,
           Runnable[Any, Other]
           | Callable[[Any], Other]
           | Any
       ], # Language model followed by optional Runnable-compatible objects
       name: str | None = None # Optional name for the resulting sequence
   ) -> RunnableSerializable[dict[str, Any], Other]
   ```


In [ ]:
from pydantic import BaseModel, Field # Import classes used to define the output schema
from langchain_core.prompts import StructuredPrompt # Import the structured prompt class
from langchain_openai import ChatOpenAI # Import a model that supports structured output

class Person(BaseModel): # Define the expected structured output
    name: str = Field(description="Name of the person") # Store the extracted person's name
    age: int = Field(description="Age of the person") # Store the extracted person's age

prompt = StructuredPrompt( # Create the structured chat prompt
    messages=[ # Define the messages used by the prompt
        ("system", "Extract the person's information."), # Provide the model instruction
        ("human", "{text}") # Insert the text supplied during invocation
    ], # Finish defining the messages
    schema_=Person, # Set the required structured output schema
    structured_output_kwargs={"method": "json_schema"} # Configure structured output generation
) # Finish creating the structured prompt

print(prompt.schema_) # Display the stored output schema
print(prompt.structured_output_kwargs) # Display the structured-output arguments
print(prompt.get_lc_namespace()) # Display the LangChain serialization namespace

model = ChatOpenAI(model="gpt-4.1-mini") # Create a compatible language model
chain = prompt | model # Pipe the prompt into the model using the stored schema

result = chain.invoke({"text": "Aarav is 24 years old."}) # Generate validated structured output
print(result) # Display the resulting Person object